# **Data Preprocessing and Featurization(Example)**

## Install the required libraries

In [ ]:
import sys
!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install seaborn
!{sys.executable} -m pip install sklearn
!{sys.executable} -m pip install matplotlib

# Analyse the California house price dataset

In this notebook, you'll use the California house price dataset to perform data analysis and preprocessing.

## Step 0: Load the required libraries

In [ ]:
import numpy as np # a software library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays.
import pandas as pd # a software library written for the Python programming language for data manipulation and analysis. In particular, it offers data structures and operations for manipulating numerical tables and time series
import matplotlib.pyplot as plt # a plotting library for the Python programming language
import seaborn as sns # Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics

# sklearn is a machine learning software library for the Python programming language. It features various classification, regression and clustering algorithms including support vector machines, random forests, gradient boosting,
# k-means and DBSCAN, and is designed to interoperate with the Python numerical and scientific libraries NumPy and SciPy.

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

# Ignore the warning message
import warnings
warnings.filterwarnings('ignore')

## Step 1: Data collecting

In [ ]:
# Load the California housing dataset
california = fetch_california_housing()

In [ ]:
# Check the description of dataset
print(california.DESCR)

In [ ]:
# Print the total row and column length of the dataset
california.data.shape

In [ ]:
# Print feature names
california.feature_names

## Step 2: Data pre-processing

In [ ]:
# Transforming the data set to Pandas' DataFrame formate (two-dimensional, size-mutable, potentially heterogeneous tabular data)
df = pd.DataFrame(california.data)

df.head()

In [ ]:
# Enter feature as the column name
df.columns = california.feature_names
df.head()

In [ ]:
# Put the target(price) in the dataset
df['Price'] = california.target

## Step 3: Data analysis

In [ ]:
# Generate descriptive statistics.
df.describe()

In [ ]:
# Check the distribution of dataset
df.hist(figsize=(12, 10), bins=100, edgecolor="black")
plt.subplots_adjust(hspace=0.7, wspace=0.4)

In [ ]:
# Pair plot between features
features = california.feature_names
features.append('Price')
grid = sns.pairplot(df[features])

From the plot, it is clear that MedInc (Median Income) and Price are correlated with each other.

In [ ]:
# Another way to plot this is using a correlation plot.
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
plt.title('Correlation Heatmap')
plt.show()

## Step 4: Feature selection

## Feature Selection Methods for Regression

Feature selection helps improve model performance by focusing only on the most relevant features.  
Here we compare three commonly used statistical methods in regression tasks.

#### Why Use Feature Selection?
- Reduces **dimensionality** by removing irrelevant variables.  
- Improves **model performance** and training efficiency.  
- Prevents **overfitting** by focusing on the most informative features.  

### 1. Correlation-based Feature Selection
- **Idea**: Compute the correlation coefficient (e.g., Pearson’s r) between each feature and the target variable.  
- **Interpretation**: Features with higher absolute correlation values are considered more predictive.  

**Advantages**
- Very fast and easy to compute.  
- Provides an intuitive measure of linear dependence.  

**Limitations**
- Captures only **linear relationships**.  
- Ignores interactions between features.  

In [ ]:
# Correlation-based feature selection
X = df[california.feature_names]
y = df['Price']

corr_matrix = X.corr()['Price'].drop('Price')

corr_df = (
    corr_matrix.abs().round(3)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'Feature', 'Price': 'Correlation'})
)

print(corr_df.reset_index(drop=True))

### 2. ANOVA F-test (Univariate Linear Regression Test)
- **Idea**: Uses an **F-test** to evaluate the linear relationship between each feature and the target.  
- **Implementation**: In scikit-learn, `SelectKBest(f_regression)` is commonly used.  

**Key Metrics**
- **F-score**: Quantifies how strongly a feature is linearly related to the target.  
  - Larger F-scores indicate greater predictive power.  
- **p-value**: Probability that the observed relationship occurred by chance.  
  - A small p-value (< 0.05) suggests the feature is statistically significant.  

**Advantages**
- Well-suited for continuous variables in regression.  
- Provides both **F-score** (strength) and **p-value** (significance).  

**Limitations**
- Assumes a **linear relationship**.  
- Cannot capture complex, non-linear patterns.  

In [ ]:
# ANOVA F-tset-based feature selection
X = df[california.feature_names]
y = df['Price']

sel = SelectKBest(f_regression)
x_new = sel.fit_transform(X, y)

scores = sel.scores_
pvalues = sel.pvalues_
feature_scores = pd.DataFrame({
    "Feature": X.columns,
    "F-score": scores,
    "p-value": pvalues
})
feature_scores = feature_scores.drop(feature_scores[feature_scores["Feature"] == "Price"].index)
feature_scores = feature_scores.round(3).sort_values(by="F-score", ascending=False)
print(feature_scores.reset_index(drop=True))

### 3. Mutual Information (MI)
- **Idea**: Measures the **amount of information** one variable provides about another, based on probability distributions.  
- **Implementation**: In scikit-learn, `mutual_info_regression` is commonly used.  

**Key Concept**
- Mutual Information comes from **information theory**.  
- It quantifies the **reduction in uncertainty** of one variable given knowledge of another.  
- If two variables are independent → MI = **0**.  
- Higher MI values indicate that the feature provides more useful information about the target.  

**Advantages**
- Captures both **linear and non-linear** relationships.  
- Does not assume a specific functional form.  

**Limitations**
- More computationally expensive than correlation or F-test.  
- Importance scores are **relative only** (no sign, just magnitude).  
- Requires sufficient data to estimate probabilities reliably.  


In [ ]:
# Mutual information-based feature selection
X = df[california.feature_names]
y = df['Price']

mi_scores = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "MI Score": mi_scores
}).sort_values(by="MI Score", ascending=False)

mi_df["MI Score"] = mi_df["MI Score"].round(3)
mi_df = mi_df[mi_df["Feature"] != "Price"]

print(mi_df.reset_index(drop=True))

### Why did Longitude rank low in Correlation/F-test but high in Mutual Information?

This phenomenon highlights the fundamental difference between **Correlation/F-test** and **Mutual Information (MI)**.

---

#### 1. Limitations of Correlation and F-test
- Pearson correlation and ANOVA F-test are designed to capture **linear relationships**.  
- They give high scores only when the target variable increases or decreases in a straight-line relationship with a feature.  
- If the relationship is **non-linear** (e.g., curved, segmented, or location-specific), these methods may assign a very low score, even if the feature is informative.

---

#### 2. What Mutual Information Captures
- Mutual Information measures **how much knowing a feature reduces the uncertainty of the target**.  
- It can capture **both linear and non-linear dependencies**.  
- A feature can have a low correlation with the target but still have a high MI score if it provides valuable information in a non-linear way.

---

#### 3. Case of Longitude in California Housing Data
- In the California housing dataset, **Longitude** does not have a simple linear relationship with house prices.  
- For example:
  - Houses near **San Francisco, Silicon Valley, and Los Angeles** are very expensive.  
  - Inland areas at the same longitude may be much cheaper.  
- This means that overall correlation with price is weak, but **Longitude strongly indicates geographic regions** that explain large differences in house prices.  

---

#### Summary
- **Correlation / F-test**: Measures *linear proportionality* → Longitude scores low.  
- **Mutual Information**: Measures *predictive information gain* (including non-linear patterns) → Longitude scores high.  

Key takeaway: A feature with weak linear correlation may still be **highly informative** in non-linear terms, which is why MI is often more powerful in complex datasets.

## cf) **Standardization & Principal Component Analysis (PCA)**

Standardisation and PCA can give you another insight into your dataset.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Standardize the data
x_scaled = StandardScaler().fit_transform(X)

In [ ]:
# 2. Apply PCA
pca = PCA()
x_pca = pca.fit_transform(x_scaled)

In [ ]:
# 3. Explained Variance Ratio
explained_variance_ratio = pca.explained_variance_ratio_

In [ ]:
# 4. Visualize the explained variance ratio
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance Ratio by Principal Component')
plt.show()

In [ ]:
# 5. Determine the number of principal components to retain
# You can choose a threshold for the explained variance ratio (e.g., 95%)
cumulative_variance = np.cumsum(explained_variance_ratio)
n_components_to_retain = np.argmax(cumulative_variance >= 0.95) + 1

In [ ]:
# 6. Analyze the loadings of the principal components
loadings = pca.components_
feature_names = x_data.columns

# Create a DataFrame for the loadings
loadings_df = pd.DataFrame(loadings, columns=feature_names)

# Print the loadings for the retained principal components
print(f"Loadings for the first {n_components_to_retain} principal components:")
print(loadings_df.iloc[:n_components_to_retain])

In [ ]:
# 7. Interpret the results
# The loadings show the correlation between the original features and the principal components.
# Features with high absolute loadings contribute more to the variance explained by that principal component.
# By examining the loadings, you can identify which features have the most significant impact on the principal components, and thus on the y_data (Price) in this case.

print(f"\nBased on the PCA analysis, the following features seem to have the most significant impact on the 'Price':")
for i in range(n_components_to_retain):
  top_features = loadings_df.iloc[i].abs().nlargest(3).index.tolist()
  print(f"Principal Component {i+1}: {', '.join(top_features)}")

While "Median Income" stands out as a primary factor on its own, from the perspective of the principal component analysis, it becomes apparent that a combination of other factors—"AveRooms, AveBedrms, Latitude"—also plays a significant role.